# Adds covariates (RUCA urban/rural, proportion Republican, SVI) to children

In [ ]:
import pandas as pd
import zipfile
import ast
import re
import json
import numpy as np
import statistics
import datetime
import matplotlib.pyplot as plt
import gzip
import os
import math

# Load data for all children in the politics data

In [ ]:
children = np.load('/share/pi/deho-pi/AFC/child_party_2018.npy', allow_pickle = True)
children = pd.DataFrame(children)
children.columns = ['patientuid', 'age', 'household_id', 'party']

In [ ]:
len(children)- sum(pd.isna(children['party']))

In [ ]:
sum(pd.isna(children['party'])), len(children) - sum(children['party'].value_counts())

In [ ]:
# put in all of the AFC data
with gzip.open('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz') as f:
    patient_baseline = pd.read_csv(f, low_memory = False, usecols=['patientuid', 'practiceid', 'gender', 'race', 'hispanic', 'raceeth', 
                                     'fips_state', 'fips_county', 'tract', 'block', 'zipcode', 'dob'])

In [ ]:
patient_baseline

In [ ]:
children = pd.DataFrame(children)
children.columns = ['patientuid', 'age', 'household_id', 'party']
children_merged = pd.merge(pd.DataFrame(children),pd.DataFrame(patient_baseline))

# Load data for everyone without a household id

In [ ]:
# load people with no household ID
no_household_ID = (
    pd.read_csv(
        '/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz',
        compression='gzip'
    )
    .loc[lambda df: df['household_id'].isna()]
)

In [ ]:
no_household_ID = no_household_ID[['patientuid', 'practiceid', 'gender', 'race', 'hispanic', 'raceeth', 
                                     'fips_state', 'fips_county', 'tract', 'block', 'zipcode', 'dob']]

In [ ]:
# combine the datasets

In [ ]:
no_household_ID['age'] = np.nan
no_household_ID['party'] = np.nan
no_household_ID['household_id'] = np.nan

In [ ]:
patients = pd.concat([children_merged, no_household_ID])

In [ ]:
patients

In [ ]:
len(children) + len(no_household_ID)

# Get zip code -> ZCTA dataset

In [ ]:
zip_zcta = pd.read_csv("../../../ZIPCodeToZCTA.csv")

zip_zcta['zipcode'] = zip_zcta['ZIP_CODE']
zip_zcta = zip_zcta[['zipcode', 'zcta']]
zip_zcta['zipcode'] = zip_zcta['zipcode'].astype('Int64')
zip_zcta = zip_zcta.drop_duplicates()

In [ ]:
patients['zipcode'] = pd.to_numeric(patients['zipcode'], errors="coerce").round(0).astype("Int64")

In [ ]:
len(patients)

In [ ]:
patients_zcta = pd.merge(patients, zip_zcta, on = 'zipcode', 
    how='left',
    indicator=True)
patients_zcta['zcta'] = patients_zcta['zcta'].astype('Int64')

In [ ]:
len(patients_zcta)

In [ ]:
# when the patient does have a state, just wasn't matched in this dataset (which happens to the below zip codes), make their recorded zcta their zipcode
not_merged = patients_zcta[patients_zcta['_merge'] == 'left_only']
not_merged[~np.isnan(not_merged.fips_state)].zipcode.value_counts()

In [ ]:
mask = (
    (patients_zcta['_merge'] == 'left_only') &
    (patients_zcta['fips_state'].notna())
)

patients_zcta.loc[mask, 'zcta'] = patients_zcta.loc[mask, 'zipcode']

In [ ]:
patients_zcta = patients_zcta[['patientuid', 'practiceid', 'age', 'household_id', 'party', 'gender', 'race',
       'hispanic', 'raceeth', 'dob', 'fips_state', 'fips_county', 'tract',
       'block', 'zcta']]

In [ ]:
len(patients_zcta)

## Add urban/rural

In [ ]:
# add urban/rural
def add_urban_rural_flag(patients_df, ruca_df):
    # Ensure tract code format matches (usually strings, padded)
    patients_df = patients_df.copy()
    #patients_df['zipcode'] = patients_df['zipcode'].astype(str).str.zfill(11)
    #ruca_df['ZIP_CODE'] = ruca_df['ZIP_CODE'].astype(str).str.zfill(11)

    merged = pd.merge(patients_zcta, ruca_df, 
                         left_on = 'zcta',
                         right_on = 'zipcode', 
                         how='left')

    def classify_ruca(code):
        if pd.isna(code):
            return 'Unknown'
        elif code <= 6:
            return 'Urban'
        elif code <= 10:
            return 'Rural'
        else:
            return 'Unknown'

    merged['urban_rural'] = merged['RUCA1'].apply(classify_ruca)
    return merged

In [ ]:
ruca_df = pd.read_csv('../../../RUCA2010zipcode.csv')
ruca_df.columns = ['ZIP_CODE', 'STATE', 'ZIP_TYPE', 'RUCA1', 'RUCA2']
ruca_df['zipcode'] = ruca_df['ZIP_CODE'].str.replace("'", "").str.zfill(5)
ruca_df = ruca_df[['zipcode', 'STATE', 'ZIP_TYPE', 'RUCA1', 'RUCA2']]
ruca_df['zipcode'] = ruca_df['zipcode'].astype('Int64')

In [ ]:
patients_ruca = add_urban_rural_flag(patients_zcta, ruca_df)

In [ ]:
len(patients_ruca)

In [ ]:
patients_ruca[pd.isna(patients_ruca['zipcode'])].STATE.value_counts()

## Add SVI

In [ ]:
# https://svi.cdc.gov/dataDownloads/data-download.html
svi = pd.read_csv('../../../SVI_2022_US_ZCTA.csv')
svi = svi[['FIPS', 'RPL_THEMES']]
svi['zipcode'] = (
    svi['FIPS']
    .astype('Int64')
)
svi['svi'] = svi.RPL_THEMES
svi = svi[['svi', 'zipcode']]

In [ ]:
patients_ruca.loc[patients_ruca['zcta'] == 80251, 'zcta'] = 80204
patients_ruca.loc[patients_ruca['zcta'] == 40366, 'zcta'] = 40360
patients_ruca.loc[patients_ruca['zcta'] == 80262, 'zcta'] = 80220
patients_ruca.loc[patients_ruca['zcta'] == 19080, 'zcta'] = 19087

In [ ]:
patients_svi = pd.merge(
    patients_ruca.drop('zipcode', axis=1),
    svi,
    left_on = 'zcta',
    right_on = 'zipcode',
    how='left')

In [ ]:
patients_svi[(np.isnan(patients_svi.svi)) & 
             (~pd.isna(patients_svi.STATE)) & 
             (patients_svi.STATE!= 'PR')]

In [ ]:
sum(patients_svi['svi'] == -999)

In [ ]:
patients_svi.loc[patients_svi['svi'] == -999, 'svi'] = np.nan

In [ ]:
sum(patients_svi['svi'] == -999)

In [ ]:
len(patients_svi), len(patients_ruca)

In [ ]:
patients_svi.columns

In [ ]:
len(patients_svi)

## Add proportion Republican at time of birth

In [ ]:
rep_prop = pd.read_csv('../../../countypres_1988_2020.csv')
rep_prop['year'] = rep_prop['year'].astype(int)

In [ ]:
rep_prop

In [ ]:
# Alaska geographies don't match -- don't allow matches
rep_prop = rep_prop[rep_prop['fips_state'] != 2]

In [ ]:
patients_svi['dob'] = pd.to_datetime(patients_svi['dob'])
patients_svi['dob_year'] = patients_svi['dob'].dt.year
patients_svi.loc[patients_svi['fips_state'] == '<NA>', 'fips_state'] = np.nan
patients_svi['fips_state'] = patients_svi['fips_state'].astype('Int64')
patients_svi.loc[patients_svi['fips_county'] == '<NA>', 'fips_county'] = np.nan
patients_svi['fips_county'] = patients_svi['fips_county'].astype('Int64')

In [ ]:
for col in ['fips_state', 'fips_county']:
    rep_prop[col] = rep_prop[col].astype(int)

In [ ]:
# create a unique child id
patients_svi = patients_svi.reset_index(drop=True)
patients_svi['child_id'] = patients_svi.index

In [ ]:
len(patients_svi['child_id'].unique())

In [ ]:
# fixing patients_svi for Colorado
patients_svi.loc[(patients_svi['tract'].isin([31201, 30400, 30000, 31202, 30900, 30100, 30300])) 
                  & (patients_svi['dob_year'] < 2001) 
                  & (patients_svi['fips_state'] == 8), 'fips_county'] = 13 # Boulder county 
                  
# 31300 is mostly Adams
patients_svi.loc[(patients_svi['tract'].isin([30500, 30700, 30800, 31000, 31401, 31300, 30600, 31403]))
                 & (patients_svi['dob_year'] < 2001) 
                 & (patients_svi['fips_state'] == 8), 'fips_county'] = 1 # Adams county

patients_svi.loc[(patients_svi['zipcode'] == 80023) 
                & (patients_svi['dob_year'] <= 2001) 
                & (patients_svi['fips_state'] == 8)
                & (np.isnan(patients_svi['tract']))
                & (patients_svi['fips_county'] == 14), 'fips_county'] = 1 # mostly Adams county

patients_svi.loc[(patients_svi['zipcode'] == 80020) 
                & (patients_svi['dob_year'] <= 2001) 
                & (patients_svi['fips_state'] == 8)
                & (np.isnan(patients_svi['tract']))
                & (patients_svi['fips_county'] == 14), 'fips_county'] = 13 # mostly Boulder county

patients_svi.loc[(patients_svi['tract'].isin([30200]))
                 & (patients_svi['dob_year'] < 2001) 
                 & (patients_svi['fips_state'] == 8), 'fips_county'] = 59 # Jefferson
                  
patients_svi.loc[(patients_svi['fips_state'] == '46') & (patients_svi['fips_county'] == '102') & (patients_svi['dob_year'] < 2015), 'fips_county'] = 113 # Oglala Lakota County used to be Shannon County

In [ ]:
merged_prez = pd.merge(patients_svi, rep_prop,
    on=['fips_state', 'fips_county'],
    how='left'
)

In [ ]:
len(merged_prez['child_id'].unique())

In [ ]:
# keep only prez years at or before dob year
merged_prez_check = merged_prez[
    (merged_prez['dob_year'] < 1988) | 
    (merged_prez['year'] <= merged_prez['dob_year']) | 
    (np.isnan(merged_prez['dob_year']))]

In [ ]:
# make sure there aren't any children that were missed but should have been counted
unmerged = merged_prez[merged_prez['child_id'].isin(set(merged_prez['child_id'])- set(merged_prez_check['child_id'])) & ~np.isnan(merged_prez['dob_year'])]


In [ ]:
# all are either in Alaska or don't have a recorded state
unmerged[['fips_state', 'fips_county']].value_counts()

In [ ]:
len(merged_prez['child_id'].unique()) # should be the correct number of unique children

In [ ]:
len(merged_prez) 

In [ ]:
# narrow to unique children
merged_prez = merged_prez[
    (merged_prez['dob_year'] < 1988) | 
    (merged_prez['year'] <= merged_prez['dob_year']) | 
    (np.isnan(merged_prez['dob_year'])) |
    (np.isnan(merged_prez['year'])) |
    (merged_prez['fips_state'] == 2)]


In [ ]:
len(merged_prez['child_id'].unique())

In [ ]:
merged_prez['effective_dob_year'] = merged_prez['dob_year'].clip(lower=1988)

merged_prez = merged_prez[
    (merged_prez['year'] <= merged_prez['effective_dob_year']) |
    (np.isnan(merged_prez['dob_year'])) |
    (np.isnan(merged_prez['year'])) |
    (merged_prez['fips_state'] == 2)]

In [ ]:
merged_prez = (
    merged_prez
    .sort_values(['child_id', 'year'], ascending=[True, False])
    .drop_duplicates(subset='child_id', keep='first')
)

In [ ]:
len(merged_prez.child_id.unique()), len(merged_prez), len(patients_svi)

In [ ]:
# check one of our changed tracts merged correctly
merged_prez[(merged_prez['tract'] == 30500) & 
            (merged_prez['dob_year'] < 2001) & 
            (merged_prez['fips_state'] == 8)]

## Add vaccine policy in the state at time of birth

In [ ]:
vax_policy_state = pd.read_csv('../../../State_vaccine_exemptions.csv')
state_to_fips = {
    "AL": 1,  "AK": 2,  "AZ": 4,  "AR": 5,  "CA": 6,  "CO": 8,
    "CT": 9,  "DE": 10, "DC": 11, "FL": 12, "GA": 13, "HI": 15,
    "ID": 16, "IL": 17, "IN": 18, "IA": 19, "KS": 20, "KY": 21,
    "LA": 22, "ME": 23, "MD": 24, "MA": 25, "MI": 26, "MN": 27,
    "MS": 28, "MO": 29, "MT": 30, "NE": 31, "NV": 32, "NH": 33,
    "NJ": 34, "NM": 35, "NY": 36, "NC": 37, "ND": 38, "OH": 39,
    "OK": 40, "OR": 41, "PA": 42, "RI": 44, "SC": 45, "SD": 46,
    "TN": 47, "TX": 48, "UT": 49, "VT": 50, "VA": 51, "WA": 53,
    "WV": 54, "WI": 55, "WY": 56
}

In [ ]:
vax_policy_state['fips_state'] = vax_policy_state['Abbreviation'].map(state_to_fips)

In [ ]:
def parse_date_range_string(date_str):
    if pd.isna(date_str):
        return []
    ranges = date_str.split(',')
    parsed_ranges = []
    for r in ranges:
        parts = r.strip().split('-')
        if len(parts) != 2:
            continue
        try:
            start = datetime.datetime.strptime(parts[0].strip(), "%m/%d/%Y")
            end = datetime.datetime.strptime(parts[1].strip(), "%m/%d/%Y")
            parsed_ranges.append((start, end))
        except ValueError:
            continue
    return parsed_ranges

def build_exemption_dataframe(vaccine_df):
    rows = []
    for _, row in vaccine_df.iterrows():
        fips = row['fips_state']
        for status, col in [
            ('religious_yes', 'Dates with \nreligious exemption'),
            ('religious_no', 'Dates without \nreligous exemption'),
            ('personal_yes', 'Dates with personal \nbeliefs exemption'),
            ('personal_no', 'Dates without personal\nbeliefs exemption')
        ]:
            ranges = parse_date_range_string(row[col])
            for start, end in ranges:
                rows.append({'fips_state': fips, 'type': status, 'start': start, 'end': end})
    return pd.DataFrame(rows)

def evaluate_exemptions_vectorized(vaccine_df, patients_df):
    exemption_df = build_exemption_dataframe(vaccine_df)
    
    patients_df = patients_df.copy()
    patients_df['dob'] = pd.to_datetime(patients_df['dob']).dt.tz_localize(None)
    patients_df = patients_df.reset_index(drop=True)
    patients_df['patient_id'] = patients_df.index
    
    merged = patients_df.merge(exemption_df, on ='fips_state', how='left')
    merged['in_range'] = (merged['dob'] >= merged['start']) & (merged['dob'] <= merged['end'])

    valid = merged[merged['in_range']].copy()

    pivot = valid.pivot_table(
        index='patient_id',
        columns='type',
        values='in_range',
        aggfunc='any'
    ).fillna(False)

    result = pd.Series(0, index=patients_df['patient_id'])

    has_religious = pivot.get('religious_yes', False) & ~pivot.get('religious_no', False)
    has_personal  = pivot.get('personal_yes', False) & ~pivot.get('personal_no', False)

    has_religious = has_religious.reindex(result.index, fill_value=False)
    has_personal = has_personal.reindex(result.index, fill_value=False)

    result[has_religious] = 1
    result[has_religious & has_personal] = 2

    return result.tolist()

In [ ]:
merged_prez['dob'] = pd.to_datetime(merged_prez['dob']).dt.tz_localize(None)

In [ ]:
child_state_exemptions = evaluate_exemptions_vectorized(vax_policy_state, merged_prez)

In [ ]:
merged_prez['policy'] = child_state_exemptions

In [ ]:
len(merged_prez)

In [ ]:
len(merged_prez.child_id.unique())

In [ ]:
# everyone outside of alaska who has a county has a share republican
merged_prez[(np.isnan(merged_prez.share_republican) & ~pd.isna(merged_prez.fips_county) & (~merged_prez.fips_state == 2))]

In [ ]:
# everyone with a state has a policy
merged_prez[(np.isnan(merged_prez.policy) & ~pd.isna(merged_prez.fips_state))]

In [ ]:
merged_prez.columns

In [ ]:
len(merged_prez.child_id.unique())

## Save

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/patients_ruca_svi_2018.csv', merged_prez)

In [ ]:
merged_prez.columns